# 21 — AutoGen Debate
## Two-Agent Turn-Based Argumentation with Microsoft AutoGen
⏱ ~45 min

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/master/examples/21-autogen-debate/autogen_debate_workbook.ipynb)

Multi-agent frameworks must solve the same core problem: **how do agents take turns?** LangGraph exposes the turn loop as an explicit graph topology you build and inspect. AutoGen hides it behind `initiate_chat()` — the framework orchestrates alternating replies so you focus on agent roles, not plumbing. This workshop teaches both the what and the why.

By the end you will:
- Understand `ConversableAgent` — the building block of every AutoGen application
- Run a two-agent debate where a Proponent and Opponent argue back and forth autonomously
- Inspect and parse `chat_history` to extract argument structure
- Tune debate quality by adjusting system prompts, turn count, and temperature
- Know exactly when to choose AutoGen vs LangGraph

---
### Workshop Roadmap
| # | Topic |
|---|-------|
| 1 | **Concepts** — AutoGen architecture, ConversableAgent, the turn loop |
| 2 | **Setup** — install, API key, imports |
| 3 | **LLM Config** — connecting agents to OpenAI |
| 4 | **First Agent** — building a ConversableAgent from scratch |
| 5 | **The Proponent** — system prompt crafting for a FOR stance |
| 6 | **The Opponent** — system prompt crafting for an AGAINST stance |
| 7 | **initiate_chat()** — running the turn loop |
| 8 | **Chat History** — parsing the returned messages |
| 9 | **Debate Display** — pretty-printing turns |
| 10 | **Full run_debate() function** — the complete workflow |
| 11 | **Multiple Topics** — swapping topics without code changes |
| 12 | **Tuning Turn Count** — MAX_TURNS tradeoffs |
| 13 | **Prompt Engineering** — making agents argue better |
| 14 | **Framework Contrast** — AutoGen vs LangGraph mental models |
| ★ | **Exercises + Answer Key** |

---
### Prerequisites
- Python 3.10+, or Google Colab
- `OPENAI_API_KEY` in `.env` or Colab Secrets
- `pyautogen` and `python-dotenv` (install cell below)

### Key References
> Wu et al. (2023) *AutoGen: Enabling Next-Gen LLM Applications via Multi-Agent Conversation* — https://arxiv.org/abs/2308.08155  
> AutoGen docs — https://microsoft.github.io/autogen/  
> ConversableAgent API — https://microsoft.github.io/autogen/docs/reference/agentchat/conversable_agent

## Part 1 — Core Concepts

### What is AutoGen?

AutoGen is Microsoft Research's multi-agent conversation framework. Its core insight: **most agent workflows are just conversations**. Instead of building an explicit graph of nodes and edges, you define agents with roles and let them talk.

```
AutoGen Mental Model
====================

  Agent A  ←──── framework manages replies ────→  Agent B
    │                                               │
    │  system_message = "You argue FOR X"           │  system_message = "You argue AGAINST X"
    │  llm_config = {model, api_key}               │  llm_config = {model, api_key}
    │  human_input_mode = "NEVER"                  │  human_input_mode = "NEVER"
    │                                               │
    └──── A.initiate_chat(B, message=...) ─────────►
             turn 1: A speaks
             turn 2: B replies
             turn 3: A speaks
             turn 4: B replies
             ... until max_turns
```

### ConversableAgent

The foundational class. Every AutoGen agent is (or inherits from) `ConversableAgent`. Key constructor arguments:

| Argument | Purpose | Common Values |
|---|---|---|
| `name` | Agent identifier, appears in chat history | `"Proponent"`, `"Opponent"` |
| `system_message` | Persona + instructions | Role description + topic stance |
| `llm_config` | LLM connection dict | `{"model": "gpt-4o-mini", "api_key": ...}` |
| `human_input_mode` | Whether to pause for human input | `"NEVER"` (autonomous), `"ALWAYS"`, `"TERMINATE"` |
| `is_termination_msg` | Callable that stops the loop early | e.g. stop when agent says `"DONE"` |

### The Turn Loop

`initiate_chat(recipient, message, max_turns)` runs the conversation:
1. Sender sends `message` to recipient
2. Recipient generates a reply via `.generate_reply()`
3. That reply becomes the sender's next input
4. Repeat until `max_turns` is reached or a termination condition fires

The key difference from LangGraph: **you don't build this loop**. It's inside AutoGen. The tradeoff is control vs convenience.

## Part 2 — Setup

In [ ]:
import sys

def _in_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False

if _in_colab():
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "pyautogen", "python-dotenv"])
    print("Colab install complete.")
else:
    print("Local — skipping install.")

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()

key = os.environ.get("OPENAI_API_KEY", "")
print(f"API key ready: {bool(key) and key.startswith('sk-')}")

## Part 3 — LLM Config

AutoGen's `llm_config` is a dict (or list of dicts for fallback chains). The minimal form needs only a `model` name and `api_key`.

**Why `gpt-4o-mini`?**  
Debate turns are short (2-3 sentences each). `gpt-4o-mini` is fast and cheap, which matters because each turn re-sends the entire conversation as context — token cost scales with `O(turns^2)`. Use `gpt-4o` when argument quality matters more than cost.

In [ ]:
import os

# The LLM config dict is passed to every ConversableAgent.
# AutoGen also supports a list of configs for fallback: if the first fails, it tries the next.
llm_config = {
    "model": "gpt-4o-mini",
    "api_key": os.environ.get("OPENAI_API_KEY"),
}

print("LLM config:")
print(f"  model   : {llm_config['model']}")
print(f"  api_key : {'set' if llm_config['api_key'] else 'MISSING'}")

## Part 4 — Building Your First ConversableAgent

A `ConversableAgent` is just a wrapper around an LLM call with:
- A name (used in `chat_history`)
- A system message (its persona and task)
- An LLM config (which model to call)
- An input mode (whether to pause for humans)

`human_input_mode="NEVER"` is what makes the agent fully autonomous — it never pauses to ask a human for input. This is the setting you want for automated pipelines.

In [ ]:
from autogen import ConversableAgent

# A minimal ConversableAgent — this is all AutoGen needs to create a speaking agent.
simple_agent = ConversableAgent(
    name="Simple",
    system_message="You are a helpful assistant. Answer concisely.",
    llm_config=llm_config,
    human_input_mode="NEVER",
)

print(f"Agent name         : {simple_agent.name}")
print(f"Human input mode   : {simple_agent.human_input_mode}")
print(f"System message     : {simple_agent.system_message[:60]}...")

## Part 5 — The Proponent Agent

Good debate agents need system prompts that:
1. Lock in their **stance** (FOR or AGAINST)
2. Inject the **topic** so they know what they're arguing about
3. Set a **response style** — concise rebuttals land better than essays
4. Optionally add **debate tactics** (address the opponent's point before making your own)

The `{topic}` is baked into the system message at construction time — agents don't adapt their stance dynamically.

In [ ]:
topic = "Remote work is more productive than in-office work"

# The system message does three things:
#   1. Assigns the stance (FOR)
#   2. Injects the topic at construction time
#   3. Constrains response length (2-3 sentences keeps turns fast and cheap)
proponent = ConversableAgent(
    name="Proponent",
    system_message=(
        f"You are a skilled debater arguing FOR: '{topic}'. "
        "Make strong, well-reasoned arguments. Be concise -- 2-3 sentences per turn."
    ),
    llm_config=llm_config,
    human_input_mode="NEVER",
)

print(f"Proponent created: {proponent.name}")
print(f"Stance: FOR '{topic}'")
print(f"System message:\n  {proponent.system_message}")

## Part 6 — The Opponent Agent

The Opponent is structurally identical to the Proponent — same class, same constructor. Only the system message differs. This symmetry is intentional: AutoGen doesn't have special "FOR" or "AGAINST" agent types, just agents with different prompts.

Notice the instruction `"Counter each argument directly"` — this creates a tighter debate than if both agents just stated their position independently.

In [ ]:
# Mirror of the Proponent — same class, opposite stance.
# "Counter each argument directly" makes the debate adversarial rather than two parallel monologues.
opponent = ConversableAgent(
    name="Opponent",
    system_message=(
        f"You are a skilled debater arguing AGAINST: '{topic}'. "
        "Counter each argument directly. Be concise -- 2-3 sentences per turn."
    ),
    llm_config=llm_config,
    human_input_mode="NEVER",
)

print(f"Opponent created: {opponent.name}")
print(f"Stance: AGAINST '{topic}'")
print(f"System message:\n  {opponent.system_message}")

print("\nBoth agents ready — same class, different prompts.")

## Part 7 — initiate_chat(): The Turn Loop

`initiate_chat()` is where AutoGen's magic happens. It:
1. Sends the opening `message` from the caller (Proponent) to the recipient (Opponent)
2. Calls `opponent.generate_reply()` — an LLM call with the Opponent's system message + conversation so far
3. Feeds the Opponent's reply back to the Proponent
4. Calls `proponent.generate_reply()` — an LLM call with the Proponent's system message + conversation so far
5. Repeats until `max_turns` is reached

**`silent=True`** suppresses printing during the run. Without it, each turn is printed to stdout in real time.

The return value is a `ChatResult` object. Its `.chat_history` attribute is a list of message dicts.

In [ ]:
# max_turns=4 means 4 complete exchanges (2 per agent).
# Each exchange is one message — so 4 turns = 4 LLM calls total.
# silent=True suppresses AutoGen's built-in printing so we control display below.
MAX_TURNS = 4

print(f"Starting debate: '{topic}'")
print(f"Max turns: {MAX_TURNS}\n")

result = proponent.initiate_chat(
    opponent,
    message=f"Proposition: '{topic}'. I argue FOR it -- let's debate.",
    max_turns=MAX_TURNS,
    silent=True,
)

print(f"Debate complete.")
print(f"ChatResult type : {type(result).__name__}")
print(f"Turns recorded  : {len(result.chat_history)}")

## Part 8 — Parsing chat_history

`chat_history` is a list of dicts. Each dict has:
- `"role"`: `"assistant"` or `"user"` (from the perspective of the calling agent)
- `"name"`: The agent's `name` attribute (e.g. `"Proponent"`, `"Opponent"`)
- `"content"`: The message text

The `name` field is more useful than `role` for display because `role` depends on which agent you're asking — the same message can be `"assistant"` to one agent and `"user"` to another.

In [ ]:
chat_history = result.chat_history

print(f"Total messages in history: {len(chat_history)}\n")
print("Raw structure of first message:")
print(chat_history[0])

print("\nKeys in each message dict:")
for key in chat_history[0].keys():
    print(f"  {key!r}")

print("\nAll speakers in this debate:")
speakers = [msg.get("name") or msg.get("role", "?") for msg in chat_history]
print(speakers)

## Part 9 — Displaying the Debate

The raw `chat_history` is useful for programmatic analysis but hard to read. A simple formatter makes the debate readable. We extract the speaker name from `"name"` (preferred) or fall back to `"role"` for any system-generated messages.

In [ ]:
def display_debate(topic: str, history: list[dict]) -> None:
    """Pretty-print a debate from chat_history."""
    print(f"\n{'=' * 60}")
    print(f"PROPOSITION: {topic}")
    print(f"{'=' * 60}\n")

    for i, msg in enumerate(history):
        speaker = msg.get("name") or msg.get("role", "unknown")
        content = msg.get("content", "")
        if not content:
            continue
        print(f"Turn {i + 1} — [{speaker.upper()}]")
        print(f"{content}")
        print()

    print(f"{'=' * 60}")
    print(f"Debate ended after {len(history)} turns.")


display_debate(topic, chat_history)

## Part 10 — The Full run_debate() Function

Now we assemble everything into the `run_debate()` function from `src/workflow.py`. This is the complete implementation — all concepts from Parts 3-9 in one composable unit.

Notice the function:
1. Takes only a `topic` string — agents are created fresh each call
2. Returns `result.chat_history` — a plain list of dicts, no AutoGen objects
3. Is stateless — calling it twice on the same topic produces different (random) results

In [ ]:
from autogen import ConversableAgent
import os

MAX_TURNS = 4


def get_llm_config() -> dict:
    """Build AutoGen LLM config dict from environment."""
    return {
        "model": "gpt-4o-mini",
        "api_key": os.environ.get("OPENAI_API_KEY"),
    }


def run_debate(topic: str) -> list[dict]:
    """
    Run a two-agent AutoGen debate on the given topic.

    Unlike LangGraph (explicit StateGraph + nodes), AutoGen agents communicate
    directly via initiate_chat -- the framework manages the turn loop internally.
    Returns the full chat history as a list of message dicts.
    """
    llm_config = get_llm_config()

    proponent = ConversableAgent(
        name="Proponent",
        system_message=(
            f"You are a skilled debater arguing FOR: '{topic}'. "
            "Make strong, well-reasoned arguments. Be concise -- 2-3 sentences per turn."
        ),
        llm_config=llm_config,
        human_input_mode="NEVER",
    )
    opponent = ConversableAgent(
        name="Opponent",
        system_message=(
            f"You are a skilled debater arguing AGAINST: '{topic}'. "
            "Counter each argument directly. Be concise -- 2-3 sentences per turn."
        ),
        llm_config=llm_config,
        human_input_mode="NEVER",
    )

    result = proponent.initiate_chat(
        opponent,
        message=f"Proposition: '{topic}'. I argue FOR it -- let's debate.",
        max_turns=MAX_TURNS,
        silent=True,
    )
    return result.chat_history


# Test it
print("run_debate() ready.")
print(f"Signature: run_debate(topic: str) -> list[dict]")

## Part 11 — Multiple Topics

The DEBATE_TOPICS list from `src/tools.py` shows how to make the debate system reusable. Changing the topic is just changing the string — agents re-read their system message every time because they're recreated in `run_debate()`.

This is different from stateful agents that "remember" previous conversations. AutoGen's `ConversableAgent` is stateless by default — each `initiate_chat()` call starts fresh.

In [ ]:
DEBATE_TOPICS = [
    "Remote work is more productive than in-office work",
    "AI will create more jobs than it eliminates",
    "Open-source software is safer than proprietary software",
]

# Run a debate on topic 1 (index 0)
chosen_topic = DEBATE_TOPICS[0]
print(f"Running debate on: {chosen_topic}\n")

history = run_debate(chosen_topic)
display_debate(chosen_topic, history)

print(f"\nOther available topics:")
for i, t in enumerate(DEBATE_TOPICS[1:], start=1):
    print(f"  DEBATE_TOPICS[{i}] = '{t}'")

## Part 12 — Tuning MAX_TURNS

`MAX_TURNS` is the most impactful parameter for cost and quality. Here is the tradeoff:

| MAX_TURNS | Exchanges per agent | API calls | Token cost | Depth |
|---|---|---|---|---|
| 2 | 1 each | 2 | Low | Opening statements only |
| 4 | 2 each | 4 | Medium | Opening + one rebuttal |
| 6 | 3 each | 6 | High | Opening + two rebuttals |
| 10 | 5 each | 10 | Very high | Full debate with closings |

**Token growth is quadratic**: turn N re-sends all N-1 previous messages as context. A 10-turn debate sends roughly 5x the tokens of a 4-turn debate.

Run the cell below to see how history length maps to turn count.

In [ ]:
def run_debate_with_turns(topic: str, max_turns: int) -> list[dict]:
    """Run a debate with a custom turn count."""
    llm_config = get_llm_config()

    proponent = ConversableAgent(
        name="Proponent",
        system_message=(
            f"You are a skilled debater arguing FOR: '{topic}'. "
            "Make strong, well-reasoned arguments. Be concise -- 2-3 sentences per turn."
        ),
        llm_config=llm_config,
        human_input_mode="NEVER",
    )
    opponent = ConversableAgent(
        name="Opponent",
        system_message=(
            f"You are a skilled debater arguing AGAINST: '{topic}'. "
            "Counter each argument directly. Be concise -- 2-3 sentences per turn."
        ),
        llm_config=llm_config,
        human_input_mode="NEVER",
    )

    result = proponent.initiate_chat(
        opponent,
        message=f"Proposition: '{topic}'. I argue FOR it -- let's debate.",
        max_turns=max_turns,
        silent=True,
    )
    return result.chat_history


# Compare 2-turn vs 4-turn debates
topic = DEBATE_TOPICS[1]  # "AI will create more jobs than it eliminates"
print(f"Topic: {topic}")

short = run_debate_with_turns(topic, max_turns=2)
long  = run_debate_with_turns(topic, max_turns=4)

print(f"\n2-turn debate: {len(short)} messages")
print(f"4-turn debate: {len(long)} messages")

print("\nFirst 2-turn Proponent argument:")
print(short[0]["content"])
print("\nFirst 4-turn Proponent argument:")
print(long[0]["content"])

## Part 13 — Prompt Engineering for Better Debates

The quality of a debate depends almost entirely on system prompt design. Three patterns that improve debate quality:

**Pattern 1 — Address before argue**  
Force the agent to acknowledge the opponent's point before making its own. This creates genuine dialogue rather than parallel monologues.

**Pattern 2 — Specific debate tactics**  
Name the rhetorical move: "Use statistics", "Give a concrete example", "Use an analogy".

**Pattern 3 — Closing statement trigger**  
Use `is_termination_msg` to detect when an agent signals they're done, then add a closing statement prompt.

In [ ]:
topic = DEBATE_TOPICS[2]  # "Open-source software is safer than proprietary software"

# Pattern 1: "Address, then argue" creates real back-and-forth
def run_structured_debate(topic: str) -> list[dict]:
    """Debate with improved prompts that force genuine engagement."""
    llm_config = get_llm_config()

    proponent = ConversableAgent(
        name="Proponent",
        system_message=(
            f"You are a skilled debater arguing FOR: '{topic}'. "
            "Structure each response as: (1) briefly acknowledge your opponent's point, "
            "(2) counter it with evidence or logic, (3) advance your own argument. "
            "Be concise -- 3-4 sentences total."
        ),
        llm_config=llm_config,
        human_input_mode="NEVER",
    )
    opponent = ConversableAgent(
        name="Opponent",
        system_message=(
            f"You are a skilled debater arguing AGAINST: '{topic}'. "
            "Structure each response as: (1) briefly acknowledge your opponent's point, "
            "(2) counter it with evidence or logic, (3) advance your own argument. "
            "Be concise -- 3-4 sentences total."
        ),
        llm_config=llm_config,
        human_input_mode="NEVER",
    )

    result = proponent.initiate_chat(
        opponent,
        message=f"Proposition: '{topic}'. I argue FOR it -- let's debate.",
        max_turns=MAX_TURNS,
        silent=True,
    )
    return result.chat_history


print(f"Running structured debate on: {topic}\n")
structured_history = run_structured_debate(topic)
display_debate(topic, structured_history)

## Part 14 — Analyzing the Debate

Once you have `chat_history`, you can analyze it programmatically. Useful metrics:
- **Turn count**: How many exchanges happened?
- **Word count per turn**: Are agents respecting the conciseness constraint?
- **Argument tracking**: What claim did each side make?

More advanced: feed the debate back into an LLM to judge who "won".

In [ ]:
def analyze_debate(history: list[dict]) -> dict:
    """Extract basic metrics from a debate history."""
    speakers = {}
    for msg in history:
        speaker = msg.get("name") or msg.get("role", "unknown")
        content = msg.get("content", "")
        if not content:
            continue
        if speaker not in speakers:
            speakers[speaker] = {"turns": 0, "total_words": 0, "arguments": []}
        speakers[speaker]["turns"] += 1
        word_count = len(content.split())
        speakers[speaker]["total_words"] += word_count
        speakers[speaker]["arguments"].append(content[:100] + "..." if len(content) > 100 else content)

    return {
        "total_turns": len(history),
        "speakers": speakers,
    }


# Analyze the structured debate from Part 13
analysis = analyze_debate(structured_history)

print(f"Total turns: {analysis['total_turns']}\n")
for speaker, stats in analysis["speakers"].items():
    avg_words = stats["total_words"] / stats["turns"] if stats["turns"] > 0 else 0
    print(f"[{speaker}]")
    print(f"  Turns       : {stats['turns']}")
    print(f"  Total words : {stats['total_words']}")
    print(f"  Avg/turn    : {avg_words:.0f} words")
    print()

## Part 15 — Judging the Debate

A powerful extension: add a third agent as a **judge**. The judge reads the full debate and decides who argued more effectively. This is a common pattern in LLM evaluation — using one LLM to score the output of other LLMs.

The judge agent receives the full debate as a string in its first message, not through `initiate_chat()` — it just needs a single `.generate_reply()` call.

In [ ]:
def judge_debate(topic: str, history: list[dict]) -> str:
    """Use a judge agent to evaluate who won the debate."""
    llm_config = get_llm_config()

    judge = ConversableAgent(
        name="Judge",
        system_message=(
            "You are an impartial debate judge. Evaluate arguments on:"
            " (1) logical strength, (2) evidence quality, (3) direct engagement with the opponent."
            " Conclude with WINNER: Proponent or WINNER: Opponent and a 1-sentence reason."
        ),
        llm_config=llm_config,
        human_input_mode="NEVER",
    )

    # Format the debate as a readable transcript for the judge
    transcript = f"TOPIC: {topic}\n\n"
    for msg in history:
        speaker = msg.get("name") or msg.get("role", "?")
        content = msg.get("content", "")
        if content:
            transcript += f"[{speaker.upper()}]: {content}\n\n"

    # Single-turn judgment — no back-and-forth needed
    messages = [{"role": "user", "content": f"Please judge this debate:\n\n{transcript}"}]
    reply = judge.generate_reply(messages=messages)
    return reply


print("Asking judge to evaluate the debate...\n")
verdict = judge_debate(topic, structured_history)
print("JUDGE'S VERDICT:")
print(verdict)

## Part 16 — Framework Contrast: AutoGen vs LangGraph

This example exists in a series that spans multiple frameworks. Understanding why you'd choose each one is as important as knowing how to use either.

```
AutoGen                              LangGraph
=======                              =========

Proponent ──► initiate_chat() ──►   START
   ↑              │                   │
   │    [framework owns loop]          ▼
   └──────────────┘               proponent_node
                                       │
                                  should_continue?
                                   /          \
                                  yes          no
                                   │            │
                              opponent_node    END
                                   │
                              should_continue?
                                   │
                              proponent_node
                                   ...
```

| Dimension | AutoGen | LangGraph |
|---|---|---|
| Turn control | Framework-managed | Explicit edges + conditionals |
| State | Implicit in chat_history | Typed State dict |
| Branching | Limited | Full graph expressiveness |
| Streaming | Built-in | Built-in |
| Debugging | Harder (hidden loop) | Easier (inspectable graph) |
| Lines of code | Fewer | More |
| Use case | Conversational agents | Complex multi-step workflows |

In [ ]:
# Demonstrate the architectural difference in code volume

autogen_approach = """
# AutoGen: ~15 lines
proponent = ConversableAgent(name="Proponent", system_message=..., llm_config=...)
opponent  = ConversableAgent(name="Opponent",  system_message=..., llm_config=...)
result = proponent.initiate_chat(opponent, message=..., max_turns=4)
history = result.chat_history
"""

langgraph_approach = """
# LangGraph: ~40+ lines
class State(TypedDict):
    messages: list
    turn: int

def proponent_node(state):
    ...  # call LLM with proponent system message

def opponent_node(state):
    ...  # call LLM with opponent system message

def should_continue(state):
    return "opponent" if state["turn"] < MAX_TURNS else END

graph = StateGraph(State)
graph.add_node("proponent", proponent_node)
graph.add_node("opponent", opponent_node)
graph.add_edge(START, "proponent")
graph.add_conditional_edges("proponent", should_continue)
graph.add_edge("opponent", "proponent")
app = graph.compile()
"""

print("AutoGen approach:", autogen_approach)
print("LangGraph approach:", langgraph_approach)
print("For a simple turn-based debate, AutoGen wins on brevity.")
print("For anything with branching, conditions, or complex state, LangGraph wins on control.")

## Part 17 — The Main Entry Point

This is how `main.py` ties everything together. It's a minimal CLI wrapper: load env, pick a topic, run the debate, print results. The separation of `main.py` (I/O) from `src/workflow.py` (logic) follows the project's convention — entry points should be thin.

In [ ]:
# This is the exact logic from main.py
# (load_dotenv() already called in the API key cell above)

def main(topic: str) -> None:
    print(f"\n{'=' * 60}")
    print(f"Proposition: {topic}")
    print(f"{'=' * 60}\n")

    history = run_debate(topic)

    for msg in history:
        name = msg.get("name") or msg.get("role", "unknown")
        content = msg.get("content", "")
        if content:
            print(f"[{name.upper()}]\n{content}\n")


main(DEBATE_TOPICS[0])

---
## Exercises

Test your understanding by completing these exercises. Each builds on the code from the workshop.

---

### Exercise 1 — Custom Topic

Run a debate on a topic of your choice. Use a topic that has a genuine FOR and AGAINST position.

Requirements:
- Define a `my_topic` string with your chosen proposition
- Run `run_debate(my_topic)` 
- Display the result with `display_debate()`
- Print the word count for each agent using `analyze_debate()`

---

### Exercise 2 — Extend to 3 Agents

Add a **Moderator** agent that asks a clarifying question after the first exchange. The moderator should:
- Have `human_input_mode="NEVER"`
- Inject one message after turn 2 with a specific clarifying question
- Not take a side

Hint: You cannot inject a moderator mid-conversation with `initiate_chat()`. Instead, manually construct the conversation by running a short debate, inspecting the history, calling `moderator.generate_reply()` with that history, then appending the moderator's message and continuing.

---

### Exercise 3 — Termination Condition

Modify `run_debate()` to stop early when either agent uses the word `"concede"`. Use the `is_termination_msg` parameter of `ConversableAgent`.

- Add `is_termination_msg=lambda msg: "concede" in msg.get("content", "").lower()` to both agents
- Run the debate
- Print whether termination was triggered or the debate ran to `MAX_TURNS`

In [ ]:
# ── EXERCISE 1 ANSWER KEY ──────────────────────────────────────────

my_topic = "Universal basic income would reduce poverty more than targeted welfare programs"

ex1_history = run_debate(my_topic)
display_debate(my_topic, ex1_history)

ex1_analysis = analyze_debate(ex1_history)
print("\nWord counts per agent:")
for speaker, stats in ex1_analysis["speakers"].items():
    print(f"  {speaker}: {stats['total_words']} words across {stats['turns']} turns")

In [ ]:
# ── EXERCISE 2 ANSWER KEY ──────────────────────────────────────────

def run_moderated_debate(topic: str) -> list[dict]:
    """Debate with a moderator who injects a clarifying question after turn 2."""
    llm_config = get_llm_config()

    proponent = ConversableAgent(
        name="Proponent",
        system_message=(
            f"You are a skilled debater arguing FOR: '{topic}'. "
            "Be concise -- 2-3 sentences per turn."
        ),
        llm_config=llm_config,
        human_input_mode="NEVER",
    )
    opponent = ConversableAgent(
        name="Opponent",
        system_message=(
            f"You are a skilled debater arguing AGAINST: '{topic}'. "
            "Counter each argument directly. Be concise -- 2-3 sentences per turn."
        ),
        llm_config=llm_config,
        human_input_mode="NEVER",
    )
    moderator = ConversableAgent(
        name="Moderator",
        system_message=(
            "You are a neutral debate moderator. Given the debate so far, "
            "ask exactly one clarifying question that will sharpen the argument. "
            "Do not take sides. One sentence only."
        ),
        llm_config=llm_config,
        human_input_mode="NEVER",
    )

    # Phase 1: first 2 turns
    result = proponent.initiate_chat(
        opponent,
        message=f"Proposition: '{topic}'. I argue FOR it -- let's debate.",
        max_turns=2,
        silent=True,
    )
    partial_history = result.chat_history

    # Moderator asks a clarifying question based on the first 2 turns
    moderator_messages = [
        {"role": "user", "content": (
            f"Here is the debate so far on '{topic}':\n"
            + "\n".join([f"[{m.get('name', m.get('role', '?'))}]: {m.get('content', '')}" for m in partial_history])
        )}
    ]
    moderator_question = moderator.generate_reply(messages=moderator_messages)

    injected = {
        "role": "user",
        "name": "Moderator",
        "content": f"[Moderator interjects]: {moderator_question}",
    }
    partial_history.append(injected)

    print("\n--- MODERATOR INTERJECTION ---")
    print(moderator_question)
    print("--- DEBATE CONTINUES ---\n")

    # Phase 2: 2 more turns
    result2 = proponent.initiate_chat(
        opponent,
        message=f"[Continuing debate, addressing moderator's question: {moderator_question}]",
        max_turns=2,
        silent=True,
    )

    return partial_history + result2.chat_history


ex2_topic = DEBATE_TOPICS[1]
print(f"Moderated debate on: {ex2_topic}\n")
ex2_history = run_moderated_debate(ex2_topic)
display_debate(ex2_topic, ex2_history)

In [ ]:
# ── EXERCISE 3 ANSWER KEY ──────────────────────────────────────────

def run_debate_with_termination(topic: str) -> tuple[list[dict], bool]:
    """Debate that stops early if either agent uses the word 'concede'."""
    llm_config = get_llm_config()

    # is_termination_msg: called after each reply to check if the conversation should stop
    def is_concession(msg: dict) -> bool:
        return "concede" in msg.get("content", "").lower()

    proponent = ConversableAgent(
        name="Proponent",
        system_message=(
            f"You are a skilled debater arguing FOR: '{topic}'. "
            "If you feel the opponent has made an irrefutable point, you may say 'I concede this point'. "
            "Otherwise, be concise -- 2-3 sentences per turn."
        ),
        llm_config=llm_config,
        human_input_mode="NEVER",
        is_termination_msg=is_concession,
    )
    opponent = ConversableAgent(
        name="Opponent",
        system_message=(
            f"You are a skilled debater arguing AGAINST: '{topic}'. "
            "If you feel the opponent has made an irrefutable point, you may say 'I concede this point'. "
            "Otherwise, counter each argument directly. Be concise -- 2-3 sentences per turn."
        ),
        llm_config=llm_config,
        human_input_mode="NEVER",
        is_termination_msg=is_concession,
    )

    result = proponent.initiate_chat(
        opponent,
        message=f"Proposition: '{topic}'. I argue FOR it -- let's debate.",
        max_turns=6,  # Higher limit to allow early termination to be meaningful
        silent=True,
    )

    history = result.chat_history
    terminated_early = any("concede" in m.get("content", "").lower() for m in history)
    return history, terminated_early


ex3_topic = DEBATE_TOPICS[2]
print(f"Running termination-aware debate on: {ex3_topic}\n")
ex3_history, ended_early = run_debate_with_termination(ex3_topic)

print(f"Turns completed : {len(ex3_history)}")
print(f"Early termination triggered: {ended_early}")

display_debate(ex3_topic, ex3_history)

---
## Workshop Complete

You have:
- Built `ConversableAgent` instances from scratch
- Run a two-agent debate with `initiate_chat()`
- Parsed and displayed `chat_history`
- Tuned debate quality through prompt engineering
- Added a judge agent for LLM-as-evaluator
- Understood the AutoGen vs LangGraph architectural tradeoff
- Extended the base pattern with moderation and termination conditions

**Key takeaway**: AutoGen's turn loop abstraction trades control for conciseness. For conversational agents with symmetric turn-taking, it is the right tool. For workflows with branching logic, typed state, or step-by-step observability, reach for LangGraph.

---
Next: **Example 22** — the next example in the series continues exploring agent patterns.

To run the production script for this example:
```bash
python examples/21-autogen-debate/main.py
```